# Step 8 — Mini RAG Demo

Combining retrieval and prompting for LLM.

In [2]:
from pathlib import Path
import sqlite3
import numpy as np
from sentence_transformers import SentenceTransformer

DB_PATH = Path("../data/linux_docs.db")

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

MODEL_NAME = "BAAI/bge-small-en-v1.5"
embedder = SentenceTransformer(MODEL_NAME)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [3]:
question = "How do I partition the disk during Arch Linux installation?"
TOP_K = 5
print(question)


How do I partition the disk during Arch Linux installation?


In [4]:
def retrieve(question: str, top_k: int = 5):
    query_vector = embedder.encode(
        question,
        normalize_embeddings=True
    ).astype(np.float32)

    cursor.execute('''
    SELECT e.chunk_id, e.embedding, c.section, c.content
    FROM embeddings e
    JOIN chunks c ON e.chunk_id = c.id
    ''')

    results = []

    for row in cursor.fetchall():
        vec = np.frombuffer(row["embedding"], dtype=np.float32)
        score = float(np.dot(query_vector, vec))

        results.append({
            "score": score,
            "section": row["section"],
            "content": row["content"]
        })

    results.sort(key=lambda x: x["score"], reverse=True)
    return results[:top_k]


In [5]:
contexts = retrieve(question, TOP_K)

for idx, item in enumerate(contexts, start=1):
    print(f"{idx}. {item['section']} ({item['score']:.4f})")


1. Introduction (0.7738)
2. Boot loader (0.7717)
3. fdisk/dev/the_disk_to_be_partitioned (0.7711)
4. Boot the live environment (0.7528)
5. Example layouts (0.7364)


In [6]:
context_text = "\n\n".join(
    f"[Section: {c['section']}]\n{c['content']}"
    for c in contexts
)

prompt = f"""You are a Linux documentation assistant.

Answer ONLY using the provided context.

If the answer is not contained in the context, say you don't know.

Context:
{context_text}

Question:
{question}

Answer:
"""

print(prompt[:4000])


You are a Linux documentation assistant.

Answer ONLY using the provided context.

If the answer is not contained in the context, say you don't know.

Context:
[Section: Introduction]
This document is a guide for installing [Arch Linux](/title/Arch_Linux) using the live system booted from an installation medium made from an official installation image. Its accessibility features are described on the page [Install Arch Linux with accessibility options](/title/Install_Arch_Linux_with_accessibility_options). For alternative means of installation, see [Category:Installation process](/title/Category:Installation_process).

Before installing, it would be advised to view the [FAQ](/title/FAQ). For conventions used in this document, see [Help:Reading](/title/Help:Reading). In particular, code examples may contain placeholders (formatted in *italics*

This guide is kept concise and you are advised to follow the instructions in the presented order per section. For more detailed instructions, see

## Send prompt to LLM (Optional)

Example using OpenAI SDK:

```python
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="gpt-4.1",
    input=prompt,
)

print(response.output_text)
```

You can also replace this section with Ollama, LM Studio, or other providers.

In [ ]:
conn.close()
print("Done ✅")

Selesai 🎉
